In [0]:
# =============================================================
# OpsIntel Copilot — Week 7
# Generate RAG Documents for Bedrock Knowledge Base
# Reads gold tables and writes text summaries to S3
# =============================================================

import boto3
import json
from datetime import datetime

BUCKET = "opsintel-copilot-angad-0025"
RAG_PREFIX = "rag-docs/"

CATALOG = "workspace"
SCHEMA  = "opsintel_copilot"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

s3 = boto3.client("s3", region_name="us-east-1")

print("✅ Setup complete")
print(f"   Bucket: {BUCKET}")
print(f"   RAG prefix: {RAG_PREFIX}")

In [0]:
def push_correlation_summaries():
    print("\n🔄 Generating correlation summaries...")
    df = spark.sql("""
        SELECT correlation_id, correlation_type, pipeline_name,
               pipeline_failed_at, error_message, security_event_type,
               security_event_at, involved_user, time_diff_minutes,
               confidence_score, recommendation
        FROM workspace.opsintel_copilot.gold_correlation_records
        WHERE confidence_score >= 0.8
        LIMIT 200
    """)
    rows = df.collect()
    print(f"   Correlations to document: {len(rows)}")
    docs = []
    for row in rows:
        doc = f"""INCIDENT CORRELATION RECORD
Correlation ID: {row['correlation_id']}
Type: {row['correlation_type']}
Pipeline: {row['pipeline_name']}
Pipeline Failed At: {row['pipeline_failed_at']}
Error: {row['error_message']}
Security Event: {row['security_event_type']}
Security Event At: {row['security_event_at']}
User Involved: {row['involved_user']}
Time Between Events: {row['time_diff_minutes']:.1f} minutes
Confidence Score: {row['confidence_score']}
Recommendation: {row['recommendation']}
---"""
        docs.append(doc)
    content = "\n".join(docs)
    s3_path = f"s3://{BUCKET}/{RAG_PREFIX}correlations/correlation_summaries.txt"
    dbutils.fs.rm(s3_path, recurse=True); dbutils.fs.put(s3_path, content, overwrite=True)
    print(f"   ✅ Pushed {len(rows)} correlations to {s3_path}")

push_correlation_summaries()

In [0]:
def push_security_summaries():
    print("\n🔄 Generating security alert summaries...")
    df = spark.sql("""
        SELECT event_id, event_type, user_email, region, event_time,
               is_brute_force, is_escalation, is_impossible_travel,
               is_large_export, is_token_rotation
        FROM workspace.opsintel_copilot.gold_security_alerts
    """)
    rows = df.collect()
    print(f"   Alerts to document: {len(rows)}")
    docs = []
    for row in rows:
        flags = []
        if row['is_brute_force']: flags.append("brute force attack")
        if row['is_escalation']: flags.append("privilege escalation")
        if row['is_impossible_travel']: flags.append("impossible travel")
        if row['is_large_export']: flags.append("large data export")
        if row['is_token_rotation']: flags.append("API token rotation")
        doc = f"""SECURITY ALERT
Alert ID: {row['event_id']}
Event Type: {row['event_type']}
User: {row['user_email']}
Region: {row['region']}
Time: {row['event_time']}
Suspicious Activities Detected: {', '.join(flags) if flags else 'none'}
---"""
        docs.append(doc)
    content = "\n".join(docs)
    s3_path = f"s3://{BUCKET}/{RAG_PREFIX}security/security_alerts.txt"
    dbutils.fs.put(s3_path, content, overwrite=True)
    print(f"   ✅ Pushed {len(rows)} alerts to {s3_path}")

push_security_summaries()

In [0]:
def push_incident_summaries():
    print("\n🔄 Generating incident summaries...")
    df = spark.sql("""
        SELECT incident_hour, region, total_events, brute_force_count,
               escalation_count, impossible_travel_count, large_export_count
        FROM workspace.opsintel_copilot.gold_incident_summary
        WHERE total_events > 5
        ORDER BY total_events DESC
        LIMIT 100
    """)
    rows = df.collect()
    print(f"   Incidents to document: {len(rows)}")
    docs = []
    for row in rows:
        doc = f"""INCIDENT SUMMARY
Time Period: {row['incident_hour']}
Region: {row['region']}
Total Security Events: {row['total_events']}
Brute Force Attempts: {row['brute_force_count']}
Privilege Escalations: {row['escalation_count']}
Impossible Travel Events: {row['impossible_travel_count']}
Large Data Exports: {row['large_export_count']}
---"""
        docs.append(doc)
    content = "\n".join(docs)
    s3_path = f"s3://{BUCKET}/{RAG_PREFIX}incidents/incident_summaries.txt"
    dbutils.fs.put(s3_path, content, overwrite=True)
    print(f"   ✅ Pushed {len(rows)} incidents to {s3_path}")

push_incident_summaries()

In [0]:
def push_playbooks():
    print("\n🔄 Pushing security playbooks...")
    playbooks = {
        "brute_force_playbook.txt": """SECURITY PLAYBOOK: Brute Force Attack Response
Detection: Multiple failed login attempts from same IP within 5 minutes
Threshold: 10+ failures triggers alert
Immediate Actions:
1. Block source IP at firewall level
2. Reset credentials for targeted account
3. Enable MFA if not already active
4. Review access logs for successful logins from same IP
5. Check if account was compromised after brute force
Investigation Questions:
- Was the attack successful? Look for login_success after login_failure events
- Is this IP seen in other regions? Could indicate distributed attack
- Did pipeline failures occur within 30 minutes? May be correlated
Escalation: If account was compromised, escalate to security team immediately""",

        "privilege_escalation_playbook.txt": """SECURITY PLAYBOOK: Privilege Escalation Response
Detection: User account granted elevated permissions outside normal change window
Risk Level: HIGH - Lateral movement and data exfiltration risk
Immediate Actions:
1. Identify what permissions were granted and to which account
2. Check if escalation was authorized via change management system
3. Review all actions taken by account after escalation
4. If unauthorized, revoke permissions immediately
Investigation Questions:
- Did any pipeline failures occur after the escalation?
- Was there a large data export following the escalation?
- Were any API tokens rotated after the escalation?
Correlation Rule: Privilege escalation + API token rotation within 10 minutes = lateral_movement""",

        "pipeline_failure_playbook.txt": """SECURITY PLAYBOOK: Pipeline Failure Investigation
When a data pipeline fails, always check for correlated security events.
Common Failure Patterns:
1. Config change before failure: Check admin_events for config_changed within 30 minutes
2. Access denied errors: Check if permissions were changed recently
3. Invalid timestamp format: Data quality issue, check source data generators
4. S3 access denied: Check IAM policy changes or S3 bucket policy modifications
Correlation Rules:
- security_to_pipeline: Admin config change within 30 min of pipeline failure
- lateral_movement: Privilege escalation + token rotation before failure
Investigation Steps:
1. Check gold_correlation_records for any correlations linked to the failed pipeline run
2. Review the timeline of events in the 2 hours before failure
3. Check if same pipeline failed recently - may indicate systematic issue
4. Review data quality results for schema drift or row count anomalies"""
    }
    for filename, content in playbooks.items():
        s3_path = f"s3://{BUCKET}/{RAG_PREFIX}playbooks/{filename}"
        dbutils.fs.put(s3_path, content, overwrite=True)
        print(f"   ✅ Pushed {s3_path}")
    print(f"\n   ✅ All playbooks pushed")

push_playbooks()

In [0]:
print("\n" + "="*60)
print("✅ RAG DOCUMENTS GENERATION COMPLETE")
print("="*60)

files = dbutils.fs.ls(f"s3://{BUCKET}/{RAG_PREFIX}")
print(f"\nFolders in s3://{BUCKET}/{RAG_PREFIX}:")
for f in files:
    print(f"   {f.path}")

print("="*60)